# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.3 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID='task117'; CH=10; H=30; W=30

ROOT=Path(COMPETITION)
if not (ROOT/f'{TASK_ID}.json').exists():
    ROOT=Path('/mnt/data')
TASK_PATH=ROOT/f'{TASK_ID}.json'

OUT_DIR=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
ONNX_PATH=OUT_DIR/f'{TASK_ID}_zero_pad_static_graph.onnx'
ZIP_PATH=OUT_DIR/f'{TASK_ID}_zero_pad_submission.zip'
STATIC_ZIP=OUT_DIR/f'{TASK_ID}_zero_pad_static_graph_submission.zip'
GENERIC_ZIP=OUT_DIR/'submission.zip'
AUDIT_JSON=OUT_DIR/f'{TASK_ID}_zero_pad_audit.json'
AUDIT_CSV=OUT_DIR/f'{TASK_ID}_zero_pad_audit.csv'

In [6]:

class Task117AxisSymmetryZeroPad(nn.Module):
    """Detects the five-cell X centre object, then completes 2-fold horizontal/vertical symmetry around its centre."""
    def __init__(self):
        super().__init__()
        Ph=torch.zeros(H,H,H)
        Pw=torch.zeros(W,W,W)
        for cc in range(H):
            for src in range(H):
                tgt=2*cc-src
                if 0<=tgt<H:
                    Ph[cc,src,tgt]=1.0
        for cc in range(W):
            for src in range(W):
                tgt=2*cc-src
                if 0<=tgt<W:
                    Pw[cc,src,tgt]=1.0
        self.register_buffer('Ph',Ph)
        self.register_buffer('Pw',Pw)
        kw=torch.zeros(9,1,3,3)
        for k in range(9):
            for i,j in [(0,0),(0,2),(1,1),(2,0),(2,2)]:
                kw[k,0,i,j]=1.0
        self.register_buffer('kw',kw)

    def reflect_rows(self,m,Mr):
        return torch.matmul(Mr.transpose(1,2).unsqueeze(1), m)

    def reflect_cols(self,m,Mc):
        return torch.matmul(m, Mc.unsqueeze(1))

    def forward(self,x):
        active=torch.clamp(x.sum(dim=1,keepdim=True),0,1)
        m=x[:,1:]
        diag=F.conv2d(m,self.kw,padding=1,groups=9)
        cnt=m.sum(dim=(2,3),keepdim=True)
        isfive=(cnt>4.5).float()*(cnt<5.5).float()
        center=torch.clamp(((diag>4.5).float()*isfive).sum(dim=1,keepdim=True),0,1)
        crow=torch.clamp(center.sum(dim=3).squeeze(1),0,1)
        ccol=torch.clamp(center.sum(dim=2).squeeze(1),0,1)
        Mr=torch.matmul(crow, self.Ph.reshape(H,H*H)).reshape(-1,H,H)
        Mc=torch.matmul(ccol, self.Pw.reshape(W,W*W)).reshape(-1,W,W)
        v=self.reflect_rows(m,Mr)
        h=self.reflect_cols(m,Mc)
        vh=self.reflect_cols(v,Mc)
        fg=torch.clamp(m+v+h+vh,0,1)*active
        fg_sum=torch.clamp(fg.sum(dim=1,keepdim=True),0,1)
        bg=active*(1.0-fg_sum)
        return torch.cat([bg,fg],dim=1)


def grid_to_tensor(grid):
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    a=np.array(grid,dtype=np.int64); h,w=a.shape
    for c in range(CH):
        x[0,c,:h,:w]=(a==c)
    return x

def tensor_to_grid(y, shape=None):
    y=np.asarray(y)
    if y.ndim==4: y=y[0]
    g=y.argmax(axis=0).astype(np.int64)
    if shape is not None: g=g[:shape[0],:shape[1]]
    return g

def exact_eval(sess, examples, raw=True):
    inp=sess.get_inputs()[0].name
    exact=0; first=None
    for i,ex in enumerate(examples):
        y=sess.run(None,{inp:grid_to_tensor(ex['input'])})[0]
        out=np.array(ex['output'],dtype=np.int64)
        pred=tensor_to_grid(y,out.shape)
        ok_arg=np.array_equal(pred,out)
        ok_raw=np.array_equal((y>0.5).astype(np.float32),grid_to_tensor(out)) if raw else True
        ok=ok_arg and ok_raw
        exact+=int(ok)
        if not ok and first is None:
            first={'idx':i,'argmax_ok':bool(ok_arg),'raw_ok':bool(ok_raw),'wrong_pixels':int((pred!=out).sum()),'shape':list(out.shape)}
    return {'exact':exact,'total':len(examples),'first_wrong':first}

def raw_contract_check(sess, examples):
    inp=sess.get_inputs()[0].name
    for i,ex in enumerate(examples):
        y=sess.run(None,{inp:grid_to_tensor(ex['input'])})[0]
        vals=set(np.unique(np.round(y,6)).tolist())
        if not vals.issubset({0.0,1.0}):
            return {'ok':False,'idx':i,'bad':'values','values':sorted(vals)[:20]}
        if not np.array_equal(y.sum(axis=1,keepdims=True), grid_to_tensor(ex['output']).sum(axis=1,keepdims=True)):
            return {'ok':False,'idx':i,'bad':'padding/channel_sum'}
    return {'ok':True}

def op_counts(model): return dict(collections.Counter(n.op_type for n in model.graph.node))
def onnx_shape(value_info): return [int(d.dim_value) if d.dim_value else None for d in value_info.type.tensor_type.shape.dim]
# Synthetic OOD examples for the same symmetry rule.
def solve_np(grid):
    a=np.array(grid,dtype=np.int64); h,w=a.shape; out=a.copy(); centers=[]
    for col in range(1,10):
        coords=np.argwhere(a==col)
        if len(coords)==5:
            S=set(map(tuple,coords))
            for r,c in coords:
                need={(r,c),(r-1,c-1),(r-1,c+1),(r+1,c-1),(r+1,c+1)}
                if need<=S: centers.append((r,c))
    if not centers: return out
    cr,cc=centers[0]
    coords=np.argwhere(a>0)
    for r,c in coords:
        col=a[r,c]
        for rr,ccol in [(2*cr-r,c),(r,2*cc-c),(2*cr-r,2*cc-c)]:
            if 0<=rr<h and 0<=ccol<w: out[rr,ccol]=col
    return out

def make_x(a,r,c,col):
    for rr,cc in [(r,c),(r-1,c-1),(r-1,c+1),(r+1,c-1),(r+1,c+1)]: a[rr,cc]=col

def synth_cases(n=500, seed=17):
    rng=random.Random(seed); cases=[]
    pats=[np.array([[1,1,0],[0,1,0],[0,0,1]]),
          np.array([[1,1,1],[0,1,0],[0,0,0]]),
          np.array([[1,1,0,1],[0,1,1,0],[1,0,1,0]])]
    tries=0
    while len(cases)<n and tries<n*200:
        tries+=1
        h=rng.randint(12,30); w=rng.randint(12,30); a=np.zeros((h,w),dtype=np.int64)
        cr=rng.randint(4,h-5); cc=rng.randint(4,w-5); cxcol=rng.randint(1,9); make_x(a,cr,cc,cxcol)
        xcoords={(cr,cc),(cr-1,cc-1),(cr-1,cc+1),(cr+1,cc-1),(cr+1,cc+1)}
        pcol=rng.choice([c for c in range(1,10) if c!=cxcol]); pat=rng.choice(pats); ph,pw=pat.shape
        top=rng.randint(1,h-ph-1); left=rng.randint(1,w-pw-1)
        base=[(top+i,left+j) for i in range(ph) for j in range(pw) if pat[i,j]]
        allpos=[]; ok=True
        for r,c in base:
            for rr,ccol in [(r,c),(2*cr-r,c),(r,2*cc-c),(2*cr-r,2*cc-c)]:
                if not (0<=rr<h and 0<=ccol<w): ok=False
                allpos.append((rr,ccol))
        if not ok: continue
        # avoid collisions with center X and between motif copies, so color priority is unambiguous.
        if len(set(allpos))!=len(allpos): continue
        if set(allpos) & xcoords: continue
        for r,c in base: a[r,c]=pcol
        cases.append({'input':a.tolist(),'output':solve_np(a).tolist()})
    return cases



In [7]:
with open(TASK_PATH) as f: task=json.load(f)
print('loaded', TASK_PATH, {k:len(v) for k,v in task.items()})

loaded /kaggle/input/competitions/neurogolf-2026/task117.json {'train': 3, 'test': 1, 'arc-gen': 261}


In [8]:
model=Task117AxisSymmetryZeroPad().eval()
dummy=torch.zeros(1,CH,H,W,dtype=torch.float32)
torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],opset_version=13,dynamic_axes=None,do_constant_folding=True,dynamo=False)

m=onnx.load(str(ONNX_PATH))
for vi in [m.graph.input[0],m.graph.output[0]]:
    for d,v in zip(vi.type.tensor_type.shape.dim,[1,CH,H,W]): d.dim_param=''; d.dim_value=int(v)
onnx.save(m,str(ONNX_PATH)); onnx.checker.check_model(m)
ops=op_counts(m)
forbidden={'Loop','Scan','NonZero','Unique','Script','Function'}
risky={'Shape','Gather','ConstantOfShape','Expand','Range','ScatterND'}
health={'task':TASK_ID,'input_shape':onnx_shape(m.graph.input[0]),'output_shape':onnx_shape(m.graph.output[0]),'size_bytes':ONNX_PATH.stat().st_size,'op_counts':ops,'forbidden_ops_present':sorted(forbidden.intersection(ops)),'risky_ops_present':sorted(risky.intersection(ops)),'accuracy':{}}
health

/tmp/ipykernel_17/1942600894.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],opset_version=13,dynamic_axes=None,do_constant_folding=True,dynamo=False)


{'task': 'task117',
 'input_shape': [1, 10, 30, 30],
 'output_shape': [1, 10, 30, 30],
 'size_bytes': 114275,
 'op_counts': {'Identity': 1,
  'Constant': 30,
  'ReduceSum': 6,
  'Clip': 6,
  'Slice': 1,
  'Conv': 1,
  'Greater': 2,
  'Cast': 3,
  'Less': 1,
  'Mul': 4,
  'Squeeze': 2,
  'MatMul': 5,
  'Reshape': 2,
  'Transpose': 1,
  'Unsqueeze': 2,
  'Add': 3,
  'Sub': 1,
  'Concat': 1},
 'forbidden_ops_present': [],
 'risky_ops_present': [],
 'accuracy': {}}

In [9]:
sess=ort.InferenceSession(str(ONNX_PATH),providers=['CPUExecutionProvider'])
for split in ['train','test','arc-gen']:
    res=exact_eval(sess,task.get(split,[]),raw=True)
    health['accuracy'][split]=f"{res['exact']}/{res['total']}"
    if res['first_wrong'] is not None: raise AssertionError((split,res))

ag=task.get('arc-gen',[])
fit_n=int(0.4*len(ag))
fit=exact_eval(sess,ag[:fit_n],raw=True)
hold=exact_eval(sess,ag[fit_n:],raw=True)
health['accuracy']['arc_gen_40_60']=f"fit {fit['exact']}/{fit['total']}, test {hold['exact']}/{hold['total']}"

synth=synth_cases(500)
sr=exact_eval(sess,synth,raw=True)
health['accuracy']['synthetic_ood']=f"{sr['exact']}/{sr['total']}"
health['raw_contract']=raw_contract_check(sess,task['train']+task['test']+ag[:20]+synth[:20])

assert health['input_shape']==[1,CH,H,W]
assert health['output_shape']==[1,CH,H,W]
assert health['size_bytes']<1_400_000, health['size_bytes']
assert not health['forbidden_ops_present'], health['forbidden_ops_present']
assert not health['risky_ops_present'], health['risky_ops_present']
assert fit['exact']==fit['total'] and hold['exact']==hold['total']
assert sr['exact']==sr['total']
assert health['raw_contract']['ok'], health['raw_contract']
health

{'task': 'task117',
 'input_shape': [1, 10, 30, 30],
 'output_shape': [1, 10, 30, 30],
 'size_bytes': 114275,
 'op_counts': {'Identity': 1,
  'Constant': 30,
  'ReduceSum': 6,
  'Clip': 6,
  'Slice': 1,
  'Conv': 1,
  'Greater': 2,
  'Cast': 3,
  'Less': 1,
  'Mul': 4,
  'Squeeze': 2,
  'MatMul': 5,
  'Reshape': 2,
  'Transpose': 1,
  'Unsqueeze': 2,
  'Add': 3,
  'Sub': 1,
  'Concat': 1},
 'forbidden_ops_present': [],
 'risky_ops_present': [],
 'accuracy': {'train': '3/3',
  'test': '1/1',
  'arc-gen': '261/261',
  'arc_gen_40_60': 'fit 104/104, test 157/157',
  'synthetic_ood': '500/500'},
 'raw_contract': {'ok': True}}

In [10]:
for zp in [ZIP_PATH, STATIC_ZIP, GENERIC_ZIP]:
    if zp.exists(): zp.unlink()
    with zipfile.ZipFile(zp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.write(ONNX_PATH,arcname=f'{TASK_ID}.onnx')
with open(AUDIT_JSON,'w') as f: json.dump(health,f,indent=2)
with open(AUDIT_CSV,'w',newline='') as f:
    wr=csv.writer(f); wr.writerow(['key','value'])
    for k,v in health.items(): wr.writerow([k,json.dumps(v) if isinstance(v,(dict,list)) else v])
print('wrote', ZIP_PATH, STATIC_ZIP, GENERIC_ZIP)

wrote /kaggle/working/task117_zero_pad_submission.zip /kaggle/working/task117_zero_pad_static_graph_submission.zip /kaggle/working/submission.zip
